In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
from matplotlib.ticker import FuncFormatter

In [3]:
rng = np.random.default_rng(42)
n = 1500

In [4]:
countries = ['United States','India','United Kingdom','South Korea','Japan','Canada','Spain','France','Germany','Brazil']
country_p = [.31,.18,.13,.10,.07,.06,.05,.04,.035,.025]

genres = ['Drama','Comedy','Documentary','Thriller','Action','Romance','Animation','Sci-Fi']
genre_p = [.24,.18,.15,.13,.11,.08,.06,.05]

types = rng.choice(['Movie','TV Show'], n, p=[.68,.32])
years = rng.triangular(1990, 2018, 2025, n).astype(int)

release_year = np.minimum(years, 2024)
added_year = np.maximum(release_year, rng.integers(2016, 2026, n))
added_year = np.minimum(added_year, 2025)

rating = rng.choice(['TV-MA','TV-14','TV-PG','R','PG-13','PG','G'], n, p=[.31,.28,.15,.12,.09,.04,.01])
country = rng.choice(countries, n, p=country_p)

genre = rng.choice(genres, n, p=genre_p)
duration = [f'{int(rng.integers(75,181))} min' if t=='Movie' else f'{int(rng.integers(1,7))} Seasons' for t in types]


In [5]:
df = pd.DataFrame({'show_id':[f's{i:05d}' for i in range(1,n+1)],'title':[f'Catalog Title {i:04d}' for i in range(1,n+1)],'type':types,'country':country,'release_year':release_year,'year_added':added_year,'rating':rating,'genre':genre,'duration':duration})
df.to_csv('netflix_sample_data.csv', index=False)
print('Created',len(df),'rows')
display(df.head())


Created 1500 rows


,show_id,title,type,country,release_year,year_added,rating,genre,duration
0,s00001,Catalog Title 0001,TV Show,India,2015,2022,TV-MA,Drama,4 Seasons
1,s00002,Catalog Title 0002,Movie,United States,2021,2024,TV-14,Comedy,116 min
2,s00003,Catalog Title 0003,TV Show,United States,2009,2022,TV-14,Action,1 Seasons
3,s00004,Catalog Title 0004,TV Show,Canada,2010,2021,R,Animation,1 Seasons
4,s00005,Catalog Title 0005,Movie,Japan,2018,2020,TV-14,Documentary,160 min


In [6]:
print('Shape:',df.shape)
print('Missing values:')
display(df.isna().sum().to_frame('missing'))
display(df.describe(include='all').T.head(9))

Shape: (1500, 9)
Missing values:


,missing
show_id,0
title,0
type,0
country,0
release_year,0
year_added,0
rating,0
genre,0
duration,0


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
show_id,1500,1500,s00001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
title,1500,1500,Catalog Title 0001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
type,1500,2,Movie,1026,NaN,NaN,NaN,NaN,NaN,NaN,NaN
country,1500,10,United States,475,NaN,NaN,NaN,NaN,NaN,NaN,NaN
release_year,1500.0,NaN,NaN,NaN,2010.520667,7.52981,1990.0,2005.0,2012.0,2017.0,2024.0
year_added,1500.0,NaN,NaN,NaN,2020.58,2.781982,2016.0,2018.0,2021.0,2023.0,2025.0
rating,1500,7,TV-MA,507,NaN,NaN,NaN,NaN,NaN,NaN,NaN
genre,1500,8,Drama,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN
duration,1500,112,5 Seasons,90,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
counts = df.type.value_counts()
by_year = df.groupby('year_added').size().reindex(range(2016,2026), fill_value=0)
by_genre = df.genre.value_counts().head(6).sort_values()
by_country = df.country.value_counts().head(6).sort_values()
by_rating = df.rating.value_counts().head(5)

In [8]:
BG='#0b1020'; PANEL='#151e32'; TEXT='#f5f7fc'; MUTED='#aab7cc'; RED='#f43f5e'; BLUE='#60a5fa'; CYAN='#22d3ee'
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'axes.titleweight':'bold','savefig.facecolor':BG})
fig = plt.figure(figsize=(16,9),facecolor=BG)
fig.text(.045,.945,'NETFLIX  |  CONTENT ANALYSIS',color=TEXT,fontsize=25,fontweight='bold')
fig.text(.045,.905,'Python • Pandas • Matplotlib   |   Synthetic portfolio dataset (not official Netflix data)',color=MUTED,fontsize=10)

Text(0.045, 0.905, 'Python • Pandas • Matplotlib   |   Synthetic portfolio dataset (not official Netflix data)')

<Figure size 1600x900 with 0 Axes>

In [9]:
cards=[('TOTAL TITLES',f'{len(df):,}'),('MOVIES',f'{counts.get("Movie",0):,}'),('TV SHOWS',f'{counts.get("TV Show",0):,}'),('COUNTRIES',str(df.country.nunique()))]
for i,(label,value) in enumerate(cards):
    x=.045+i*.233
    ax=fig.add_axes([x,.745,.213,.122]); ax.set_facecolor(PANEL)
    for s in ax.spines.values():s.set_visible(False)
    ax.set_xticks([]);ax.set_yticks([])
    ax.text(.06,.70,label,color=MUTED,fontsize=10,transform=ax.transAxes,fontweight='bold')
    ax.text(.06,.17,value,color=TEXT,fontsize=25,transform=ax.transAxes,fontweight='bold')
    ax.axhline(0,color=RED if i==0 else BLUE,linewidth=5)

In [10]:
def panel(rect,title):
    ax=fig.add_axes(rect);ax.set_facecolor(PANEL)
    for s in ax.spines.values():s.set_visible(False)
    ax.set_title(title,loc='left',color=TEXT,pad=17,fontsize=12)
    ax.tick_params(colors=MUTED,labelsize=9,length=0)
    return ax

In [11]:
ax=panel([.045,.415,.43,.268],'Titles Added by Year')
ax.plot(by_year.index,by_year.values,color=RED,marker='o',linewidth=2.7,markersize=5)
ax.fill_between(by_year.index,by_year.values,color=RED,alpha=.12)
ax.set_xticks([2016,2018,2020,2022,2024,2025]);ax.grid(axis='y',color='white',alpha=.10)
ax.set_xlim(2015.7,2025.3)

(2015.7, 2025.3)

In [12]:
ax=panel([.515,.415,.44,.268],'Top Genres')
ax.barh(by_genre.index,by_genre.values,color=RED,height=.63)
ax.set_xlim(0,by_genre.max()*1.20);ax.set_xticks([])
for y,v in enumerate(by_genre.values):ax.text(v+3,y,str(v),color=TEXT,va='center',fontsize=9)

In [13]:
ax=panel([.045,.075,.27,.265],'Movies vs TV Shows')
ax.pie([counts.get('Movie',0),counts.get('TV Show',0)],colors=[RED,BLUE],startangle=90,counterclock=False,wedgeprops={'width':.35,'edgecolor':PANEL,'linewidth':3})
ax.text(0,0,f'{counts.get("Movie",0)/n:.0%}',color=TEXT,ha='center',va='center',fontsize=19,fontweight='bold')
ax.legend(['Movies','TV Shows'],loc='lower center',bbox_to_anchor=(.5,-.10),ncol=2,frameon=False,labelcolor=MUTED,fontsize=9)

In [14]:
ax=panel([.35,.075,.305,.265],'Top Countries')
ax.barh(by_country.index,by_country.values,color=BLUE,height=.63)
ax.set_xlim(0,by_country.max()*1.17);ax.set_xticks([])
for y,v in enumerate(by_country.values):ax.text(v+3,y,str(v),color=TEXT,va='center',fontsize=8)

In [15]:
ax=panel([.69,.075,.265,.265],'Content Ratings')
ax.bar(by_rating.index,by_rating.values,color=[RED,BLUE,CYAN,'#a78bfa','#fb923c'],width=.65)
ax.grid(axis='y',color='white',alpha=.10);ax.set_axisbelow(True)
for x,v in enumerate(by_rating.values):ax.text(x,v+5,str(v),color=TEXT,ha='center',fontsize=9)

In [16]:
fig.text(.045,.026,'DATA NOTE  •  1,500 artificially generated records for educational use; all titles and statistics are illustrative.',color=MUTED,fontsize=9)
fig.savefig('netflix_one_page_dashboard.png',dpi=180,facecolor=BG,bbox_inches='tight',pad_inches=.22)
plt.show()
print('Saved netflix_one_page_dashboard.png')


Saved netflix_one_page_dashboard.png


In [20]:
from IPython.display import FileLink, display
display(FileLink('netflix_one_page_dashboard.png'))
display(FileLink('netflix_sample_data.csv'))

C:\Users\srini\netflix_one_page_dashboard.png

C:\Users\srini\netflix_sample_data.csv